# Exp 8 — H-adapter: does sharing between renderings fall with the cost of converting them to canonical form?

**Claim.** Latent tree z, rendering maps g. Sharing s(g) between g(z) and the canonical rendering decreases with a(g), the cost of converting g(z) into canonical(z) using only the model's own primitives.
Both quantities are measured with the SAME architecture (12-layer causal RoPE transformer, d=32 by default), independently of each other:

- **a(g)** — a translator trained on `[BOS g(z) SEP canonical(z)]`, loss on the canonical part only, fixed step budget. Primary = area under the output-CE curve minus the identity (pure copy) translator's. Secondary = held-out output CE / accuracy.
- **s(g)** — joint model on {A canonical 70%, A' identical rendering with a new tag 15%, g 15%} vs a DISJOINT-grammar control with identical data volume. `gain_L = CE_L(disjoint) - CE_L(shared)`; `s(g) = gain_g / gain_A'`. A' is the ceiling (same order, same tree).

**Transformations (10).** relabel_new (bijection onto NEW token ids), relabel_perm (bijection onto the SAME ids), marker, swap5, rev6, rev5, rev4, rev45, rev_all, perm_sym. All verified injective, and g(z) determines canonical(z).

**Written down before running (do not edit after seeing data):**
- **Gate (instrument):** mean gain_A' >= 0.02 nats AND >= 3 SE. If not, the instrument cannot see sharing at this d; try smaller d. Result is UNTESTABLE, not negative.
- **P1:** Spearman rho(a, s) <= -0.7 over the 10 transformations, seed-averaged.  **Dead if rho > -0.5.**  Between: inconclusive.
- **P2:** relabel_new has smaller a and larger s than relabel_perm.
- **P3 (capacity interaction, only if >1 d is run):** s rises with d more steeply for high-a than low-a transformations.
- No ordering of a(g) across transformations is assumed in advance; the translator measures it.

**Corrections to what I said in chat.** (1) "Relabeling is free because embeddings absorb it" only holds with NEW token ids; relabel_perm is the conflicting version, and both are included. (2) The "even-in-offset position encoding" causal arm is invalid for a causal model: a causal head only looks backward, so offsets are always positive and reversal does not flip their sign. It is dropped; P3 replaces it.

**Known weak points.** n = 10 points gives a noisy Spearman. Gain mixes shared grammar knowledge with shared machinery; the disjoint control removes data-volume effects but not that. The translator is a proxy for adapter cost, not the cost itself. Eval trees are small (<=160 leaves) so every rendering fits in 256 tokens. Untested here: all torch code (no GPU in my sandbox).


## 0a. RUN THIS CELL FIRST. It writes `exp8_lib.py` into the notebook's folder on the server (the launcher runs it as a script). No file upload needed.

In [1]:
%%writefile exp8_lib.py
"""Exp 8 library: adapter-cost hypothesis (H-adapter).
Claim: sharing between a rendering g(z) and the canonical rendering falls with a(g), the cost of converting g(z) into canonical(z)
using only the model's own primitives (causal RoPE attention + MLP).
  a(g): train the SAME architecture as a translator   [BOS g(z) SEP canonical(z)], loss on the canonical part only, fixed step budget.
  s(g): joint model on {A (canonical, 70%), A' (identical rendering, new tag, 15%), g (15%)}; compare against a DISJOINT-grammar control
        with the same data volume.  gain_L = CE_L(disjoint) - CE_L(shared)  (positive = sharing the tree helps language L);
        s(g) = gain_g / gain_A'   (A' is the ceiling: same order, same tree).
CLI:  python exp8_lib.py joint --g rev45 --cond shared --seed 0 --d 32 --steps 90000 --root ./exp8 --device cuda:0
      python exp8_lib.py translator --g rev45 --seed 0 --d 32 --steps 24000 --root ./exp8 --device cuda:0
Everything above the TORCH marker is torch-free (and unit-tested)."""
import os, sys, math, json, time, random, argparse
import numpy as np
from collections import defaultdict

# ------------------------------------------------------------------ grammar (verbatim from nb6) ------------------------------------------------------------------
LEVELS = 6; NT_SIZE = 4; TERM_SIZE = 4
MIN_DEGREE, MAX_DEGREE, RULE_LEN_CHOICES = 2, 4, [2, 3]
GRAMMAR_SEED = 0            # same grammar as nb6's check_rules
SEQ_LEN = 256

# token layout: identical for EVERY condition so parameter counts match
TERM_NEW0 = 4               # relabel_new terminals: 4..7
MARKER = 8
BOS = {"A": 9, "A2": 10, "G": 11}
SEP = 12
VOCAB = 13
LANGS3 = ["A", "A2", "G"]
MIX = {"A": 0.70, "A2": 0.15, "G": 0.15}

def build_grammar(seed):
    rng = random.Random(seed)
    symbols_by_level = {1: [(1, 0)]}
    for l in range(2, LEVELS + 1): symbols_by_level[l] = [(l, i) for i in range(NT_SIZE)]
    terminals = list(range(TERM_SIZE)); rules = defaultdict(list)
    for l in range(1, LEVELS + 1):
        child_level = l + 1
        child_symbols = terminals if child_level == LEVELS + 1 else symbols_by_level[child_level]
        for sym in symbols_by_level[l]:
            degree = rng.randint(MIN_DEGREE, MAX_DEGREE); seen = set(); tries = 0
            while len(rules[sym]) < degree and tries < 200:
                tries += 1; rule_len = rng.choice(RULE_LEN_CHOICES)
                rhs = tuple(rng.choice(child_symbols) for _ in range(rule_len))
                if rhs in seen: continue
                seen.add(rhs); rules[sym].append(rhs)
    return rules, symbols_by_level

def permute_grammar(rules, symbols_by_level, seed):
    rng = random.Random(seed); new_rules = {(1, 0): rules[(1, 0)]}
    for level, syms in symbols_by_level.items():
        if level == 1: continue
        shuffled = syms[:]; rng.shuffle(shuffled)
        for original_sym, new_owner in zip(syms, shuffled): new_rules[new_owner] = rules[original_sym]
    return new_rules

def sample_tree(rules, rng, symbol=(1, 0), level=1):
    rhs = rng.choice(rules[symbol]); children = []
    for child_sym in rhs:
        if level == LEVELS: children.append(child_sym)
        else: children.append(sample_tree(rules, rng, symbol=child_sym, level=level + 1))
    return (symbol, children)

# ------------------------------------------------------------------ renderings g ------------------------------------------------------------------
def _perm_table(seed):
    """fixed per-symbol child permutation, one for each (symbol, rule length); never the identity"""
    rng = random.Random(seed); tab = {}
    for l in range(1, LEVELS + 1):
        for i in range(1 if l == 1 else NT_SIZE):
            for n in (2, 3):
                p = list(range(n))
                while p == list(range(n)): rng.shuffle(p)
                tab[((l, i), n)] = p
    return tab

TRANSFORMS = {
    "identity":     {},                                                 # translator reference (pure copy)
    "relabel_new":  dict(term_map={t: TERM_NEW0 + t for t in range(TERM_SIZE)}),   # bijection onto NEW token ids: embeddings can align them
    "relabel_perm": dict(term_map={0: 1, 1: 2, 2: 3, 3: 0}),                        # bijection onto the SAME ids: conflicts in a shared embedding table
    "marker":       dict(marker=MARKER),                                # a marker token after every level-6 span
    "swap5":        dict(swap={5}),                                     # first two children swapped at level 5 only
    "rev6": dict(rev={6}), "rev5": dict(rev={5}), "rev4": dict(rev={4}), "rev45": dict(rev={4, 5}),
    "rev_all":      dict(rev=set(range(1, LEVELS + 1))),                # = reverse of the canonical string
    "perm_sym":     dict(perm_sym=_perm_table(777)),                    # parse-dependent fixed reorder, per symbol
}
G_LIST = [g for g in TRANSFORMS if g != "identity"]

def render(node, spec):
    """node from sample_tree: bare int terminal, or ((level, idx), children)."""
    if isinstance(node, int): return [spec.get("term_map", {}).get(node, node)]
    (lvl, idx), children = node
    order = list(range(len(children)))
    if lvl in spec.get("rev", ()): order = order[::-1]
    elif lvl in spec.get("swap", ()) and len(order) >= 2: order[0], order[1] = order[1], order[0]
    elif "perm_sym" in spec: order = spec["perm_sym"][((lvl, idx), len(children))]
    out = []
    for j in order: out.extend(render(children[j], spec))
    if spec.get("marker") is not None and lvl == LEVELS: out.append(spec["marker"])
    return out

def gate_injective(g, n=20000, seed=5):
    """g(z) must determine canonical(z) (else the translator target is not a function of its input) and be injective on trees."""
    rules, _ = build_grammar(GRAMMAR_SEED); rng = random.Random(seed); spec = TRANSFORMS[g]
    g2c, c2g = {}, {}; bad = 0
    for _ in range(n):
        t = sample_tree(rules, rng); gs = tuple(render(t, spec)); cs = tuple(render(t, {}))
        if g2c.setdefault(gs, cs) != cs or c2g.setdefault(cs, gs) != gs: bad += 1
    return dict(g=g, n=n, conflicts=bad, unique_g=len(g2c), unique_canon=len(c2g))

# ------------------------------------------------------------------ data ------------------------------------------------------------------
def make_grammars(cond, names=LANGS3):
    shared, sbl = build_grammar(GRAMMAR_SEED)
    if cond == "shared": return {n: shared for n in names}
    assert cond == "disjoint"
    return {n: permute_grammar(shared, sbl, 200 + i) for i, n in enumerate(names)}

def make_stream_chunk(grams, specs, mix, n_trees, seed):
    rng = random.Random(seed); names = list(mix); w = [mix[n] for n in names]; out = []
    for _ in range(n_trees):
        lang = rng.choices(names, weights=w)[0]; tree = sample_tree(grams[lang], rng)
        out.append(BOS[lang]); out.extend(render(tree, specs[lang]))
    return np.asarray(out, dtype=np.int16)

def build_pool(grams, specs, mix, big_n, chunk, data_seed):
    parts = [make_stream_chunk(grams, specs, mix, min(chunk, big_n - s), data_seed * 1000 + i) for i, s in enumerate(range(0, big_n, chunk))]
    s = np.concatenate(parts); n = len(s) // (SEQ_LEN + 1)
    return s[: n * (SEQ_LEN + 1)].reshape(n, SEQ_LEN + 1)

def make_eval_rows(grammar, spec, bos, n_trees, seed, max_leaves=160, row_len=SEQ_LEN):
    """One tree per row, [BOS, tokens..., 0 padding]. Small trees only (<= max_leaves canonical leaves) so every rendering fits in row_len."""
    rng = random.Random(seed); rows = np.zeros((n_trees, row_len), np.int16); nreal = np.zeros(n_trees, np.int64); k = 0
    while k < n_trees:
        t = sample_tree(grammar, rng)
        if len(render(t, {})) > max_leaves: continue
        toks = [bos] + render(t, spec); assert len(toks) <= row_len, len(toks)
        rows[k, : len(toks)] = toks; nreal[k] = len(toks); k += 1
    return rows, nreal

def make_translator_rows(grammar, spec, n_trees, seed, max_leaves=120, row_len=320):
    """row = [BOS_G, g(z)..., SEP, canonical(z)...]; ostart = index of the first canonical token. Loss only on canonical tokens."""
    rng = random.Random(seed); rows = np.zeros((n_trees, row_len), np.int16); ostart = np.zeros(n_trees, np.int64); rlen = np.zeros(n_trees, np.int64); k = 0
    while k < n_trees:
        t = sample_tree(grammar, rng); c = render(t, {})
        if len(c) > max_leaves: continue
        g = render(t, spec); row = [BOS["G"]] + g + [SEP] + c; assert len(row) <= row_len, len(row)
        rows[k, : len(row)] = row; ostart[k] = len(g) + 2; rlen[k] = len(row); k += 1
    return rows, ostart, rlen

def joint_paths(root, g, cond, d, seed): b = os.path.join(root, "joint", f"{g}_{cond}_d{d}_s{seed}"); return b + ".json", b + ".pt"
def tr_paths(root, g, d, seed): return os.path.join(root, "translator", f"{g}_d{d}_s{seed}.json")

# =========================================================== TORCH ===========================================================
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    TORCH = True
except Exception:
    TORCH = False

if TORCH:
    def rotate_half(x):
        x1, x2 = x.chunk(2, dim=-1); return torch.cat((-x2, x1), dim=-1)
    def apply_rotary(q, k, cos, sin): return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)
    class RotaryEmbedding(nn.Module):
        def __init__(self, dim, max_seq_len=1024, base=10000):
            super().__init__()
            inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim)); t = torch.arange(max_seq_len).float()
            freqs = torch.einsum("i,j->ij", t, inv_freq); emb = torch.cat((freqs, freqs), dim=-1)
            self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False); self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)
        def forward(self, seq_len): return self.cos[:, :, :seq_len, :], self.sin[:, :, :seq_len, :]
    class CausalSelfAttention(nn.Module):
        def __init__(self, n_embd, n_head, max_seq_len):
            super().__init__(); assert n_embd % n_head == 0
            self.n_head = n_head; self.head_dim = n_embd // n_head
            self.qkv = nn.Linear(n_embd, 3 * n_embd); self.proj = nn.Linear(n_embd, n_embd); self.rotary = RotaryEmbedding(self.head_dim, max_seq_len)
        def forward(self, x):
            B, T, C = x.shape; qkv = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4); q, k, v = qkv[0], qkv[1], qkv[2]
            cos, sin = self.rotary(T); q, k = apply_rotary(q, k, cos.to(x.dtype), sin.to(x.dtype))
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True); return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))
    class Block(nn.Module):
        def __init__(self, n_embd, n_head, max_seq_len):
            super().__init__(); self.ln1 = nn.LayerNorm(n_embd); self.attn = CausalSelfAttention(n_embd, n_head, max_seq_len); self.ln2 = nn.LayerNorm(n_embd)
            self.mlp = nn.Sequential(nn.Linear(n_embd, 4 * n_embd), nn.GELU(), nn.Linear(4 * n_embd, n_embd))
        def forward(self, x): x = x + self.attn(self.ln1(x)); return x + self.mlp(self.ln2(x))
    class TinyRotaryGPT(nn.Module):
        def __init__(self, vocab_size, n_embd, n_layer=12, n_head=4, max_seq_len=512):
            super().__init__(); assert n_embd % n_head == 0 and (n_embd // n_head) % 2 == 0, "head_dim must be even for RoPE"
            self.tok_emb = nn.Embedding(vocab_size, n_embd); self.blocks = nn.ModuleList([Block(n_embd, n_head, max_seq_len) for _ in range(n_layer)])
            self.ln_f = nn.LayerNorm(n_embd); self.head = nn.Linear(n_embd, vocab_size, bias=False)
        def forward(self, idx, targets=None):
            x = self.tok_emb(idx)
            for blk in self.blocks: x = blk(x)
            logits = self.head(self.ln_f(x)); loss = None
            if targets is not None: loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            return logits, loss

    def _atomic_json(path, obj):
        os.makedirs(os.path.dirname(path), exist_ok=True); tmp = path + ".tmp"; json.dump(obj, open(tmp, "w")); os.replace(tmp, path)

    @torch.no_grad()
    def eval_ce_per_tree(model, rows, nreal, device, bs=128):
        """per-tree mean next-token CE over in-tree targets, fp32"""
        model.eval(); out = []; ar = torch.arange(rows.shape[1] - 1, device=device)
        for i in range(0, rows.shape[0], bs):
            x = torch.from_numpy(rows[i:i + bs].astype(np.int64)).to(device); nr = torch.from_numpy(nreal[i:i + bs]).to(device)
            logits, _ = model(x); ce = F.cross_entropy(logits[:, :-1].reshape(-1, logits.shape[-1]).float(), x[:, 1:].reshape(-1), reduction="none").view(x.shape[0], -1)
            m = (ar[None, :] <= (nr - 2)[:, None]).float(); out.append(((ce * m).sum(1) / m.sum(1)).cpu())
        return torch.cat(out).numpy()

    def train_joint(g, cond, seed, d, steps, root, device, n_head=4, n_layer=12, big_n=500_000, chunk=50_000, eval_n=1000, batch=32, lr=3e-3, log_every=5000):
        rpath, cpath = joint_paths(root, g, cond, d, seed)
        if os.path.exists(rpath): print("done already:", rpath); return
        dev = torch.device(device); grams = make_grammars(cond); specs = {"A": {}, "A2": {}, "G": TRANSFORMS[g]}
        ds = seed if seed != 0 else 1; t0 = time.time()
        pool = torch.from_numpy(build_pool(grams, specs, MIX, big_n, chunk, ds)); print(f"pool {tuple(pool.shape)} in {(time.time() - t0) / 60:.1f} min", flush=True)
        torch.manual_seed(seed); model = TinyRotaryGPT(VOCAB, d, n_layer, n_head, SEQ_LEN + 1).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, fused=(dev.type == "cuda")); data = pool.to(dev); n = data.shape[0]
        buf = torch.zeros(steps, device=dev); ac = (lambda: torch.autocast(device_type="cuda", dtype=torch.bfloat16)) if dev.type == "cuda" else None
        import contextlib
        for step in range(steps):
            b = data[torch.randint(0, n, (batch,), device=dev)].long(); x, y = b[:, :-1], b[:, 1:]
            with (ac() if ac else contextlib.nullcontext()): _, loss = model(x, y)
            opt.zero_grad(); loss.backward(); opt.step(); buf[step] = loss.detach().float()
            if (step + 1) % log_every == 0: print(f"step {step + 1:>7d} loss {buf[step - log_every + 1:step + 1].mean().item():.4f}", flush=True)
        torch.save({"model_state": model.state_dict(), "step": steps, "g": g, "cond": cond, "d": d, "seed": seed}, cpath)
        res = dict(g=g, cond=cond, d=d, seed=seed, steps=steps, tail_loss=float(buf[-1000:].mean()), ce={}, ce_se={}, n_eval=eval_n)
        for lang in LANGS3:                                             # same eval seed for every language in a condition -> paired trees in the shared condition
            rows, nreal = make_eval_rows(grams[lang], specs[lang], BOS[lang], eval_n, seed=424242)
            c = eval_ce_per_tree(model, rows, nreal, dev); res["ce"][lang] = float(c.mean()); res["ce_se"][lang] = float(c.std(ddof=1) / len(c) ** 0.5)
        _atomic_json(rpath, res); print(json.dumps(res), flush=True)

    def train_translator(g, seed, d, steps, root, device, n_head=4, n_layer=12, pool_n=120_000, eval_n=1000, batch=32, lr=3e-3, win=500, row_len=320):
        rpath = tr_paths(root, g, d, seed)
        if os.path.exists(rpath): print("done already:", rpath); return
        dev = torch.device(device); rules, _ = build_grammar(GRAMMAR_SEED); spec = TRANSFORMS[g]; ds = seed if seed != 0 else 1
        rows, ostart, rlen = make_translator_rows(rules, spec, pool_n, 5000 + ds, row_len=row_len)
        R = torch.from_numpy(rows).to(dev); OS = torch.from_numpy(ostart).to(dev); RL = torch.from_numpy(rlen).to(dev)
        torch.manual_seed(seed); model = TinyRotaryGPT(VOCAB, d, n_layer, n_head, row_len + 1).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, fused=(dev.type == "cuda")); ar = torch.arange(row_len - 1, device=dev)
        import contextlib
        ac = (lambda: torch.autocast(device_type="cuda", dtype=torch.bfloat16)) if dev.type == "cuda" else None
        def masked(ce, os_, rl): return ((ar[None, :] + 1 >= os_[:, None]) & (ar[None, :] + 1 < rl[:, None])).float()
        curve = []; acc = 0.0; cnt = 0
        for step in range(steps):
            idx = torch.randint(0, R.shape[0], (batch,), device=dev); b = R[idx].long(); x, y = b[:, :-1], b[:, 1:]
            with (ac() if ac else contextlib.nullcontext()): logits, _ = model(x)
            ce = F.cross_entropy(logits.reshape(-1, logits.shape[-1]).float(), y.reshape(-1), reduction="none").view(batch, -1)
            m = masked(ce, OS[idx], RL[idx]); loss = (ce * m).sum() / m.sum()
            opt.zero_grad(); loss.backward(); opt.step(); acc += loss.item(); cnt += 1
            if cnt == win: curve.append(acc / cnt); acc = 0.0; cnt = 0; print(f"step {step + 1:>6d} out-CE {curve[-1]:.4f}", flush=True) if (len(curve) % 10 == 0) else None
        erows, eos, erl = make_translator_rows(rules, spec, eval_n, 777777, row_len=row_len)       # held-out trees
        model.eval(); tot = 0.0; tok = 0; hit = 0
        with torch.no_grad():
            for i in range(0, eval_n, 64):
                b = torch.from_numpy(erows[i:i + 64].astype(np.int64)).to(dev); x, y = b[:, :-1], b[:, 1:]; logits, _ = model(x)
                ce = F.cross_entropy(logits.reshape(-1, logits.shape[-1]).float(), y.reshape(-1), reduction="none").view(b.shape[0], -1)
                m = masked(ce, torch.from_numpy(eos[i:i + 64]).to(dev), torch.from_numpy(erl[i:i + 64]).to(dev))
                tot += (ce * m).sum().item(); tok += m.sum().item(); hit += ((logits.argmax(-1) == y).float() * m).sum().item()
        res = dict(g=g, d=d, seed=seed, steps=steps, curve=curve, auc=float(np.mean(curve)), final_train=float(np.mean(curve[-4:])),
                   heldout_ce=tot / tok, heldout_acc=hit / tok, win=win)
        _atomic_json(rpath, res); print(json.dumps({k: v for k, v in res.items() if k != "curve"}), flush=True)

    def main():
        ap = argparse.ArgumentParser(); sub = ap.add_subparsers(dest="cmd", required=True)
        for name in ("joint", "translator"):
            p = sub.add_parser(name); p.add_argument("--g", required=True, choices=list(TRANSFORMS)); p.add_argument("--seed", type=int, default=0)
            p.add_argument("--d", type=int, default=32); p.add_argument("--steps", type=int, default=90000 if name == "joint" else 24000)
            p.add_argument("--root", default="./exp8"); p.add_argument("--device", default="cuda:0"); p.add_argument("--n_head", type=int, default=4)
            if name == "joint": p.add_argument("--cond", choices=["shared", "disjoint"], required=True); p.add_argument("--big_n", type=int, default=500_000)
        a = ap.parse_args()
        if a.cmd == "joint": train_joint(a.g, a.cond, a.seed, a.d, a.steps, a.root, a.device, n_head=a.n_head, big_n=a.big_n)
        else: train_translator(a.g, a.seed, a.d, a.steps, a.root, a.device, n_head=a.n_head)

    if __name__ == "__main__": main()


Overwriting exp8_lib.py


## 0b. Config and switches

In [2]:
import os, sys, json, time, subprocess, itertools, importlib
import numpy as np
import exp8_lib as X
importlib.reload(X)

ROOT = "./exp8"
D_LIST = [32]                    # n_head=4 -> d multiple of 8. Add 16 / 64 later for P3. Smaller d = more capacity-bound = more room for sharing to matter.
SEEDS  = [0]
JOINT_STEPS = 90_000             # fixed budget, same for every joint model (nb6 lesson: never compare unequal step counts)
TRANS_STEPS = 24_000             # same for every translator
BIG_N = 500_000                  # trees per joint pool (nb6 big-pool); lower to 300_000 to save generation time
DEVICES = ["cuda:2"]             # list of GPUs to round-robin over, e.g. ["cuda:0","cuda:1","cuda:2"]
MAX_PAR = 6

LAUNCH_TRANSLATORS = True
LAUNCH_JOINT = True
os.makedirs(os.path.join(ROOT, "logs"), exist_ok=True)
print("transformations:", X.G_LIST)
print(f"jobs at these settings: translators {(len(X.G_LIST)+1)*len(D_LIST)*len(SEEDS)}, joint {2*len(X.G_LIST)*len(D_LIST)*len(SEEDS)}")


transformations: ['relabel_new', 'relabel_perm', 'marker', 'swap5', 'rev6', 'rev5', 'rev4', 'rev45', 'rev_all', 'perm_sym']
jobs at these settings: translators 11, joint 20


## 1. Gate 0 — renderings are injective (CPU, run first)

In [3]:
# Gate 0 (CPU, ~1 min): every rendering must be injective and g(z) must determine canonical(z); otherwise the translator target is ill-defined.
for g in X.G_LIST:
    r = X.gate_injective(g, n=5000)
    print(("OK  " if r["conflicts"] == 0 and r["unique_g"] == r["unique_canon"] else "BAD ") + str(r))


OK  {'g': 'relabel_new', 'n': 5000, 'conflicts': 0, 'unique_g': 5000, 'unique_canon': 5000}
OK  {'g': 'relabel_perm', 'n': 5000, 'conflicts': 0, 'unique_g': 5000, 'unique_canon': 5000}
OK  {'g': 'marker', 'n': 5000, 'conflicts': 0, 'unique_g': 5000, 'unique_canon': 5000}
OK  {'g': 'swap5', 'n': 5000, 'conflicts': 0, 'unique_g': 5000, 'unique_canon': 5000}
OK  {'g': 'rev6', 'n': 5000, 'conflicts': 0, 'unique_g': 5000, 'unique_canon': 5000}
OK  {'g': 'rev5', 'n': 5000, 'conflicts': 0, 'unique_g': 5000, 'unique_canon': 5000}
OK  {'g': 'rev4', 'n': 5000, 'conflicts': 0, 'unique_g': 5000, 'unique_canon': 5000}
OK  {'g': 'rev45', 'n': 5000, 'conflicts': 0, 'unique_g': 5000, 'unique_canon': 5000}
OK  {'g': 'rev_all', 'n': 5000, 'conflicts': 0, 'unique_g': 5000, 'unique_canon': 5000}
OK  {'g': 'perm_sym', 'n': 5000, 'conflicts': 0, 'unique_g': 5000, 'unique_canon': 5000}


## 2. Launch (resumable). Set the switches in 0b, then run. Translators are cheap; run them first to see a(g) early.

In [ ]:
def job_cmds():
    jobs = []
    for d in D_LIST:
        for seed in SEEDS:
            if LAUNCH_TRANSLATORS:
                for g in ["identity"] + X.G_LIST:
                    if not os.path.exists(X.tr_paths(ROOT, g, d, seed)):
                        jobs.append((f"tr_{g}_d{d}_s{seed}", ["translator", "--g", g, "--seed", str(seed), "--d", str(d), "--steps", str(TRANS_STEPS)]))
            if LAUNCH_JOINT:
                for g in X.G_LIST:
                    for cond in ("shared", "disjoint"):
                        if not os.path.exists(X.joint_paths(ROOT, g, cond, d, seed)[0]):
                            jobs.append((f"jt_{g}_{cond}_d{d}_s{seed}", ["joint", "--g", g, "--cond", cond, "--seed", str(seed), "--d", str(d), "--steps", str(JOINT_STEPS), "--big_n", str(BIG_N)]))
    return jobs

def run_parallel(jobs, max_par=MAX_PAR):
    running, n_started = [], 0
    print(f"{len(jobs)} jobs queued, {max_par} at a time on {DEVICES}", flush=True)
    while jobs or running:
        still = []
        for p, name, log in running:
            rc = p.poll()
            if rc is None: still.append((p, name, log))
            else: log.close(); print(f"finished {name} exit={rc}" + ("" if rc == 0 else "   <-- check the log"), flush=True)
        running = still
        while jobs and len(running) < max_par:
            name, argv = jobs.pop(0); dev = DEVICES[n_started % len(DEVICES)]; n_started += 1
            log = open(os.path.join(ROOT, "logs", name + ".txt"), "w")
            running.append((subprocess.Popen([sys.executable, "exp8_lib.py"] + argv + ["--root", ROOT, "--device", dev], stdout=log, stderr=subprocess.STDOUT), name, log))
        time.sleep(10)

# Runs in a background thread so the cell returns; the kernel must stay alive. Shutting it down orphans nothing already started, and finished jobs are skipped on relaunch.
# Each job is resumable at job granularity (finished jobs are skipped by their result json).
import threading
jobs = job_cmds()
if jobs:
    LAUNCH_THREAD = threading.Thread(target=run_parallel, args=(jobs,), daemon=False); LAUNCH_THREAD.start()   # returns immediately; keep the kernel alive
else: print("nothing to launch (switches off, or everything already finished)")


25 jobs queued, 6 at a time on ['cuda:2']


finished tr_rev5_d32_s0 exit=-15   <-- check the log
finished tr_rev4_d32_s0 exit=-15   <-- check the log
finished tr_rev45_d32_s0 exit=-15   <-- check the log
finished tr_rev_all_d32_s0 exit=-15   <-- check the log
finished tr_perm_sym_d32_s0 exit=-15   <-- check the log
finished jt_relabel_new_shared_d32_s0 exit=-15   <-- check the log


In [5]:
def status():
    for d in D_LIST:
        tr = sum(os.path.exists(X.tr_paths(ROOT, g, d, s)) for g in ["identity"] + X.G_LIST for s in SEEDS)
        jt = sum(os.path.exists(X.joint_paths(ROOT, g, c, d, s)[0]) for g in X.G_LIST for c in ("shared", "disjoint") for s in SEEDS)
        print(f"d={d}: translators {tr}/{(len(X.G_LIST)+1)*len(SEEDS)}   joint {jt}/{2*len(X.G_LIST)*len(SEEDS)}")
status()


d=32: translators 6/11   joint 0/20


## 3. Analysis: gate, P1, P2 (works on partial results)

In [6]:
import glob
def load_all(root):
    J, T = {}, {}
    for p in glob.glob(os.path.join(root, "joint", "*.json")):
        r = json.load(open(p)); J[(r["g"], r["cond"], r["d"], r["seed"])] = r
    for p in glob.glob(os.path.join(root, "translator", "*.json")):
        r = json.load(open(p)); T[(r["g"], r["d"], r["seed"])] = r
    return J, T

GATE_MIN_GAIN, GATE_Z = 0.02, 3.0
def gains(J, d, seeds):
    """per (g, seed): gain_L = CE_L(disjoint) - CE_L(shared), its SE, and s = gain_G / gain_A2 (nan unless gain_A2 clears the gate)."""
    out = {}
    for g in X.G_LIST:
        for s in seeds:
            a, b = J.get((g, "shared", d, s)), J.get((g, "disjoint", d, s))
            if a is None or b is None: continue
            gain = {L: b["ce"][L] - a["ce"][L] for L in X.LANGS3}; se = {L: (a["ce_se"][L] ** 2 + b["ce_se"][L] ** 2) ** 0.5 for L in X.LANGS3}
            ok = gain["A2"] > max(GATE_MIN_GAIN, GATE_Z * se["A2"])
            out[(g, s)] = dict(gain=gain, se=se, s=gain["G"] / gain["A2"] if ok else float("nan"), anchor_ok=ok)
    return out

def cost_a(T, d, seeds):
    """a(g) = translator AUC (and held-out CE) minus the identity translator's, per seed, then averaged"""
    out = {}
    for g in X.G_LIST:
        auc, ce, acc = [], [], []
        for s in seeds:
            t, i = T.get((g, d, s)), T.get(("identity", d, s))
            if t is None or i is None: continue
            auc.append(t["auc"] - i["auc"]); ce.append(t["heldout_ce"] - i["heldout_ce"]); acc.append(t["heldout_acc"])
        if auc: out[g] = dict(auc=float(np.mean(auc)), ce=float(np.mean(ce)), acc=float(np.mean(acc)), n=len(auc))
    return out

def _ranks(x):
    x = np.asarray(x, float); order = np.argsort(x); r = np.empty(len(x)); i = 0
    while i < len(x):
        j = i
        while j + 1 < len(x) and x[order[j + 1]] == x[order[i]]: j += 1
        r[order[i:j + 1]] = (i + j) / 2 + 1; i = j + 1
    return r
def spearman(x, y):
    rx, ry = _ranks(x), _ranks(y); return float(np.corrcoef(rx, ry)[0, 1])
def perm_p_le(x, y, rho, n=20000, seed=0):
    rng = np.random.RandomState(seed); y = np.asarray(y); return float(np.mean([spearman(x, rng.permutation(y)) <= rho for _ in range(n)]))

def report(root=ROOT, d=None, seeds=SEEDS):
    d = D_LIST[0] if d is None else d; J, T = load_all(root); G = gains(J, d, seeds); A = cost_a(T, d, seeds)
    print(f"=== d={d}, seeds={seeds} ===")
    anchors = [v["gain"]["A2"] for v in G.values()]; ase = [v["se"]["A2"] for v in G.values()]
    if anchors:
        m = float(np.mean(anchors)); sem = float(np.std(anchors, ddof=1) / len(anchors) ** 0.5) if len(anchors) > 1 else float("nan")
        print(f"GATE: gain_A' mean {m:.4f} nats (SEM over runs {sem:.4f}, typical eval SE {np.mean(ase):.4f}); runs passing per-run gate: {sum(v['anchor_ok'] for v in G.values())}/{len(G)}")
        print("  " + ("PASS: instrument sees the ceiling." if m >= GATE_MIN_GAIN and m >= GATE_Z * max(np.mean(ase) / len(anchors) ** 0.5, 1e-9) else "FAIL -> UNTESTABLE at this d (not a negative result). Try smaller d / check the shared-vs-disjoint pair."))
    rows = []
    for g in X.G_LIST:
        ss = [G[(g, s)]["s"] for s in seeds if (g, s) in G]; gg = [G[(g, s)]["gain"]["G"] for s in seeds if (g, s) in G]
        if g in A and ss: rows.append((g, A[g]["auc"], A[g]["ce"], A[g]["acc"], float(np.mean(gg)), float(np.nanmean(ss)) if np.isfinite(ss).any() else float("nan"), ss))
    rows.sort(key=lambda r: r[1])
    print(f"\n{'g':13s} {'a_auc':>8s} {'a_ce':>7s} {'acc':>6s} | {'gain_G':>7s} {'s mean':>7s}   s per seed")
    for g, a, ce, acc, gg, sm, ss in rows: print(f"{g:13s} {a:8.4f} {ce:7.4f} {acc:6.3f} | {gg:7.4f} {sm:7.2f}   " + " ".join(f"{x:5.2f}" for x in ss))
    ok = [r for r in rows if np.isfinite(r[5])]
    if len(ok) >= 4:
        a = [r[1] for r in ok]; s = [r[5] for r in ok]; rho = spearman(a, s); p = perm_p_le(a, s, rho)
        loo = [spearman([x for j, x in enumerate(a) if j != i], [x for j, x in enumerate(s) if j != i]) for i in range(len(ok))]
        print(f"\nP1  rho(a, s) = {rho:+.2f}  (n={len(ok)}, one-sided perm p = {p:.3f}, leave-one-out range [{min(loo):+.2f}, {max(loo):+.2f}])")
        print("    " + ("SUPPORTS H-adapter (rho <= -0.7)" if rho <= -0.7 else "DEAD (rho > -0.5)" if rho > -0.5 else "INCONCLUSIVE (-0.7 < rho <= -0.5)"))
        d_ = {r[0]: r for r in ok}
        if "relabel_new" in d_ and "relabel_perm" in d_:
            n, q = d_["relabel_new"], d_["relabel_perm"]
            print(f"P2  relabel_new vs relabel_perm: a {n[1]:.4f} vs {q[1]:.4f}, s {n[5]:.2f} vs {q[5]:.2f}  ->  " + ("as predicted" if n[1] < q[1] and n[5] > q[5] else "NOT as predicted"))
        try:
            import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
            plt.figure(figsize=(6, 4.5)); plt.scatter(a, s)
            for r in ok: plt.annotate(r[0], (r[1], r[5]), fontsize=8)
            plt.xlabel("a(g): translator AUC minus identity"); plt.ylabel("s(g) = gain_g / gain_A'"); plt.title(f"d={d}, rho={rho:+.2f}"); plt.tight_layout()
            plt.savefig(os.path.join(root, f"scatter_d{d}.png"), dpi=140); print("saved", os.path.join(root, f"scatter_d{d}.png"))
        except Exception as e: print("plot skipped:", e)
        return dict(rho=rho, p=p, rows=rows)
    print("\nneed >= 4 transformations with finite s and a to compute rho"); return dict(rows=rows)

def report_P3(root=ROOT, seeds=SEEDS):
    J, T = load_all(root); ds = sorted({k[2] for k in J}); 
    if len(ds) < 2: print("P3 needs >= 2 d values"); return
    A = cost_a(T, ds[0], seeds); med = float(np.median([v["auc"] for v in A.values()])); hi = [g for g in A if A[g]["auc"] > med]; lo = [g for g in A if A[g]["auc"] <= med]
    curves = {}
    for name, grp in (("high-a", hi), ("low-a", lo)):
        ys = []
        for d in ds:
            G = gains(J, d, seeds); v = [G[(g, s)]["s"] for g in grp for s in seeds if (g, s) in G and np.isfinite(G[(g, s)]["s"])]; ys.append(float(np.mean(v)) if v else float("nan"))
        curves[name] = ys; print(f"{name:7s} s by d {ds}: " + "  ".join(f"{y:.2f}" for y in ys))
    xs = np.log2(ds); sl = {k: float(np.polyfit(xs, v, 1)[0]) for k, v in curves.items() if np.isfinite(v).all()}
    print("slopes (s per doubling of d):", sl, "->", "P3 holds" if len(sl) == 2 and sl["high-a"] > sl["low-a"] else "P3 does not hold / undetermined")

R = report()


=== d=32, seeds=[0] ===

g                a_auc    a_ce    acc |  gain_G  s mean   s per seed

need >= 4 transformations with finite s and a to compute rho


## 4. P3 capacity interaction (needs >1 d in D_LIST)

In [7]:
report_P3()

P3 needs >= 2 d values


In [8]:
status()
!ls exp8/logs | head
!tail -n 3 exp8/logs/tr_identity_d32_s0.txt
!nvidia-smi

d=32: translators 6/11   joint 0/20
jt_relabel_new_shared_d32_s0.txt
tr_identity_d32_s0.txt
tr_marker_d32_s0.txt
tr_perm_sym_d32_s0.txt
tr_relabel_new_d32_s0.txt
tr_relabel_perm_d32_s0.txt
tr_rev45_d32_s0.txt
tr_rev4_d32_s0.txt
tr_rev5_d32_s0.txt
tr_rev6_d32_s0.txt
step  15000 out-CE 0.0031
step  20000 out-CE 0.0006
{"g": "identity", "d": 32, "seed": 0, "steps": 24000, "auc": 0.02240758047375124, "final_train": 0.0003437405536811582, "heldout_ce": 2.158932873193648e-06, "heldout_acc": 1.0, "win": 500}
Thu Oct 29 12:57:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|

In [9]:
!tail -n 2 exp8/logs/tr_identity_d32_s0.txt

step  20000 out-CE 0.0006
{"g": "identity", "d": 32, "seed": 0, "steps": 24000, "auc": 0.02240758047375124, "final_train": 0.0003437405536811582, "heldout_ce": 2.158932873193648e-06, "heldout_acc": 1.0, "win": 500}


In [10]:
!tail -n 2 exp8/logs/tr_identity_d32_s0.txt

step  20000 out-CE 0.0006
{"g": "identity", "d": 32, "seed": 0, "steps": 24000, "auc": 0.02240758047375124, "final_train": 0.0003437405536811582, "heldout_ce": 2.158932873193648e-06, "heldout_acc": 1.0, "win": 500}


In [11]:
!ps -eo pid,etime,args | grep exp8_lib.py | grep -v grep

2640861       07:55 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev5 --seed 0 --d 32 --steps 24000 --root ./exp8 --device cuda:2
2640862       07:55 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev4 --seed 0 --d 32 --steps 24000 --root ./exp8 --device cuda:2
2640863       07:55 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev45 --seed 0 --d 32 --steps 24000 --root ./exp8 --device cuda:2
2640864       07:55 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev_all --seed 0 --d 32 --steps 24000 --root ./exp8 --device cuda:2
2640865       07:55 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g perm_sym --seed 0 --d 32 --steps 24000 --root ./exp8 --device cuda:2
2640866       07:55 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g relabel_new --cond shared --seed 0 --d 32 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2


In [12]:
%%writefile exp8_queue.py
import argparse, os, sys, time, subprocess
import exp8_lib as X

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="./exp8"); ap.add_argument("--d", type=int, nargs="+", default=[32]); ap.add_argument("--seeds", type=int, nargs="+", default=[0, 7, 13])
    ap.add_argument("--devices", nargs="+", default=["cuda:0"]); ap.add_argument("--max_par", type=int, default=6)
    ap.add_argument("--translators", action="store_true"); ap.add_argument("--joint", action="store_true")
    ap.add_argument("--joint_steps", type=int, default=90000); ap.add_argument("--trans_steps", type=int, default=24000); ap.add_argument("--big_n", type=int, default=500000)
    ap.add_argument("--poll", type=float, default=10.0)
    a = ap.parse_args(); os.makedirs(os.path.join(a.root, "logs"), exist_ok=True)
    pidf = os.path.join(a.root, "scheduler.pid")
    if os.path.exists(pidf):
        try: os.kill(int(open(pidf).read()), 0); print("a scheduler is already running (pid", open(pidf).read(), ") - not starting a second one", flush=True); return
        except Exception: pass
    open(pidf, "w").write(str(os.getpid()))
    jobs = []
    for d in a.d:
        for seed in a.seeds:
            if a.translators:
                for g in ["identity"] + X.G_LIST:
                    jobs.append((f"tr_{g}_d{d}_s{seed}", ["translator", "--g", g, "--seed", str(seed), "--d", str(d), "--steps", str(a.trans_steps)], X.tr_paths(a.root, g, d, seed)))
            if a.joint:
                for g in X.G_LIST:
                    for cond in ("shared", "disjoint"):
                        jobs.append((f"jt_{g}_{cond}_d{d}_s{seed}", ["joint", "--g", g, "--cond", cond, "--seed", str(seed), "--d", str(d), "--steps", str(a.joint_steps), "--big_n", str(a.big_n)], X.joint_paths(a.root, g, cond, d, seed)[0]))
    n_started = 0; print(f"{len(jobs)} jobs in the queue (finished ones are skipped), max_par={a.max_par}, devices={a.devices}", flush=True)
    own = []
    while True:
        own = [(p, n) for p, n in own if p.poll() is None]
        ps = subprocess.run(["ps", "-eo", "args"], capture_output=True, text=True).stdout.splitlines()
        running_lines = [l for l in ps if "exp8_lib.py" in l and "exp8_queue.py" not in l]
        jobs = [j for j in jobs if not os.path.exists(j[2])]
        if not jobs and not running_lines: print("all jobs finished", flush=True); break
        free = a.max_par - len(running_lines)
        for j in list(jobs):
            if free <= 0: break
            name, argv, _ = j; key = "exp8_lib.py " + " ".join(argv)
            if any(key in l for l in running_lines): jobs.remove(j); continue
            dev = a.devices[n_started % len(a.devices)]; n_started += 1; free -= 1; jobs.remove(j)
            log = open(os.path.join(a.root, "logs", name + ".txt"), "a")
            p = subprocess.Popen([sys.executable, "exp8_lib.py"] + argv + ["--root", a.root, "--device", dev], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
            own.append((p, name)); print(time.strftime("%H:%M:%S"), "started", name, "on", dev, "pid", p.pid, flush=True)
        time.sleep(a.poll)
    os.remove(pidf)

if __name__ == "__main__": main()

Writing exp8_queue.py


In [13]:
import subprocess, sys
cmd = [sys.executable, "exp8_queue.py", "--root", "./exp8", "--d", "32", "--seeds", "0", "7", "13",
       "--devices", "cuda:2", "--max_par", "6", "--translators", "--joint"]
p = subprocess.Popen(cmd, stdout=open("exp8/scheduler.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
print("scheduler pid", p.pid)

scheduler pid 2681274


In [14]:
!tail -n 5 exp8/scheduler.log
!ps -eo pid,etime,args | grep -E "exp8_lib|exp8_queue" | grep -v grep

93 jobs in the queue (finished ones are skipped), max_par=6, devices=['cuda:2']
2640861       09:00 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev5 --seed 0 --d 32 --steps 24000 --root ./exp8 --device cuda:2
2640862       09:00 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev4 --seed 0 --d 32 --steps 24000 --root ./exp8 --device cuda:2
2640863       09:00 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev45 --seed 0 --d 32 --steps 24000 --root ./exp8 --device cuda:2
2640864       09:00 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g rev_all --seed 0 --d 32 --steps 24000 --root ./exp8 --device cuda:2
2640865       09:00 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py translator --g perm_sym --seed 0 --d 32 --steps 24000 --root ./exp8 --device cuda:2
2640866       09:00 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g relabel_new --cond shared --seed 0 --d 32 --steps 90000 --

In [15]:
import subprocess, os, time
subprocess.run(["pkill", "-f", "exp8_queue.py"]); time.sleep(1)
subprocess.run(["pkill", "-f", "exp8_lib.py"]);   time.sleep(3)
if os.path.exists("exp8/scheduler.pid"): os.remove("exp8/scheduler.pid")
print(subprocess.run("ps -eo pid,args | grep -E 'exp8_lib|exp8_queue' | grep -v grep",
                     shell=True, capture_output=True, text=True).stdout or "nothing running")

2686772 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g relabel_new --cond disjoint --seed 0 --d 32 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2
2686773 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g relabel_perm --cond shared --seed 0 --d 32 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2
2686774 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g relabel_perm --cond disjoint --seed 0 --d 32 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2
2686775 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g marker --cond shared --seed 0 --d 32 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2
2686776 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g marker --cond disjoint --seed 0 --d 32 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2
2686777 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g swap5 --cond shared --seed 0 --d 32 --steps 90000 --big_n 500000 

In [16]:
import subprocess, sys
cmd = [sys.executable, "exp8_queue.py", "--root", "./exp8", "--d", "32", "--seeds", "0",
       "--devices", "cuda:2", "--max_par", "6", "--translators", "--joint"]
p = subprocess.Popen(cmd, stdout=open("exp8/scheduler.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
print("scheduler pid", p.pid)

scheduler pid 2687426


In [17]:
!tail -n 8 exp8/scheduler.log
!ps -eo pid,etime,args | grep -E "exp8_lib|exp8_queue" | grep -v grep

93 jobs in the queue (finished ones are skipped), max_par=6, devices=['cuda:2']
31 jobs in the queue (finished ones are skipped), max_par=6, devices=['cuda:2']
2686772       31:52 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g relabel_new --cond disjoint --seed 0 --d 32 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2
2686773       31:52 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g relabel_perm --cond shared --seed 0 --d 32 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2
2686774       31:52 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g relabel_perm --cond disjoint --seed 0 --d 32 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2
2686775       31:52 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g marker --cond shared --seed 0 --d 32 --steps 90000 --big_n 500000 --root ./exp8 --device cuda:2
2686776       31:52 /home/anwoy/miniconda3/envs/ke/bin/python exp8_lib.py joint --g marker --cond d

In [18]:
print(len(jobs), LAUNCH_THREAD.is_alive())
jobs.clear()
print("old queue cleared:", len(jobs))

13 True
old queue cleared: 0


In [19]:
!ls exp8/translator exp8/joint
!grep -l Traceback exp8/logs/*.txt | head
!ls exp8/logs | wc -l

ls: cannot access 'exp8/joint': No such file or directory
exp8/translator:
identity_d32_s0.json  relabel_new_d32_s0.json	rev6_d32_s0.json
marker_d32_s0.json    relabel_perm_d32_s0.json	swap5_d32_s0.json
18


In [20]:
!nproc
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

128
index, memory.used [MiB], memory.total [MiB]
0, 85320 MiB, 143771 MiB
1, 96884 MiB, 143771 MiB
2, 54270 MiB, 143771 MiB
3, 135037 MiB, 143771 MiB
4, 132661 MiB, 143771 MiB
5, 117120 MiB, 143771 MiB


In [21]:
import subprocess, sys, os, time
subprocess.run(["pkill", "-f", "exp8_queue.py"]); time.sleep(2)
if os.path.exists("exp8/scheduler.pid"): os.remove("exp8/scheduler.pid")
cmd = [sys.executable, "exp8_queue.py", "--root", "./exp8", "--d", "32", "--seeds", "0",
       "--devices", "cuda:2", "--max_par", "11", "--translators", "--joint"]
p = subprocess.Popen(cmd, stdout=open("exp8/scheduler.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
print("scheduler pid", p.pid)

scheduler pid 2826143


In [29]:
!tail -n 8 exp8/scheduler.log
!ps -eo pid,etime,args | grep exp8_lib | grep -v grep | wc -l

93 jobs in the queue (finished ones are skipped), max_par=6, devices=['cuda:2']
31 jobs in the queue (finished ones are skipped), max_par=6, devices=['cuda:2']
31 jobs in the queue (finished ones are skipped), max_par=11, devices=['cuda:2']
13:42:44 started tr_rev5_d32_s0 on cuda:2 pid 2826229
13:42:44 started tr_rev4_d32_s0 on cuda:2 pid 2826230
13:42:44 started tr_rev45_d32_s0 on cuda:2 pid 2826231
13:42:44 started tr_rev_all_d32_s0 on cuda:2 pid 2826232
13:42:44 started tr_perm_sym_d32_s0 on cuda:2 pid 2826233
11


In [28]:
import glob, json
T = {}
for p in glob.glob("exp8/translator/*_s0.json"):
    r = json.load(open(p)); T[r["g"]] = r
i = T["identity"]
for g, r in sorted(T.items(), key=lambda kv: kv[1]["auc"]):
    print(f"{g:13s} auc {r['auc']:.4f}  (minus identity {r['auc']-i['auc']:+.4f})  heldout_ce {r['heldout_ce']:.4f}  acc {r['heldout_acc']:.3f}")

identity      auc 0.0224  (minus identity +0.0000)  heldout_ce 0.0000  acc 1.000
relabel_perm  auc 0.0257  (minus identity +0.0033)  heldout_ce 0.0002  acc 1.000
marker        auc 0.0265  (minus identity +0.0041)  heldout_ce 0.0008  acc 1.000
relabel_new   auc 0.0303  (minus identity +0.0079)  heldout_ce 0.0010  acc 1.000
rev6          auc 0.0309  (minus identity +0.0085)  heldout_ce 0.0022  acc 0.999
swap5         auc 0.0621  (minus identity +0.0396)  heldout_ce 0.0231  acc 0.991


In [30]:
import random, numpy as np, exp8_lib as X

def _uniq(node, c):
    if isinstance(node, int): c[0] += 1; return 1000 + c[0] - 1
    return (node[0], [_uniq(ch, c) for ch in node[1]])

def string_baselines(g, n_trees=300, seed=3):
    rules, _ = X.build_grammar(X.GRAMMAR_SEED); rng = random.Random(seed); spec = X.TRANSFORMS[g]
    disp, tau, ins = [], [], []
    for _ in range(n_trees):
        t = _uniq(X.sample_tree(rules, rng), [0]); out = X.render(t, spec); leaves = np.array([x - 1000 for x in out if x >= 1000]); n = len(leaves)
        pos = np.empty(n, int); pos[leaves] = np.arange(n)
        disp.append(np.abs(pos - np.arange(n)).mean() / n)
        tau.append(((pos[:, None] > pos[None, :]) & (np.arange(n)[:, None] < np.arange(n)[None, :])).sum() / (n * (n - 1) / 2))
        ins.append((len(out) - n) / n)
    tm = spec.get("term_map"); tokchg = float(np.mean([tm[k] != k for k in tm])) if tm else 0.0
    return dict(displacement=float(np.mean(disp)), kendall_tau=float(np.mean(tau)), inserted=float(np.mean(ins)), token_change=tokchg)

def partial_spearman(x, y, z):
    r = lambda v: np.argsort(np.argsort(np.asarray(v, float))).astype(float)
    rx, ry, rz = r(x), r(y), r(z); res = lambda a, b: a - np.polyval(np.polyfit(b, a, 1), b)
    return float(np.corrcoef(res(rx, rz), res(ry, rz))[0, 1])

BASE = {g: string_baselines(g) for g in X.G_LIST}
print(f"{'g':13s} {'displacement':>12s} {'kendall_tau':>11s} {'inserted':>9s} {'token_change':>12s}")
for g, b in BASE.items(): print(f"{g:13s} {b['displacement']:12.3f} {b['kendall_tau']:11.3f} {b['inserted']:9.3f} {b['token_change']:12.1f}")

g             displacement kendall_tau  inserted token_change
relabel_new          0.000       0.000     0.000          1.0
relabel_perm         0.000       0.000     0.000          1.0
marker               0.000       0.000     0.407          0.0
swap5                0.009       0.009     0.000          0.0
rev6                 0.005       0.007     0.000          0.0
rev5                 0.014       0.018     0.000          0.0
rev4                 0.033       0.039     0.000          0.0
rev45                0.034       0.057     0.000          0.0
rev_all              0.500       1.000     0.000          0.0
perm_sym             0.348       0.595     0.000          0.0


In [2]:
import glob, json, random, numpy as np
import exp8_lib as X

def _uniq(node, c):
    if isinstance(node, int): c[0] += 1; return 1000 + c[0] - 1
    return (node[0], [_uniq(ch, c) for ch in node[1]])

def string_baselines(g, n_trees=300, seed=3):
    rules, _ = X.build_grammar(X.GRAMMAR_SEED); rng = random.Random(seed); spec = X.TRANSFORMS[g]
    disp, tau, ins = [], [], []
    for _ in range(n_trees):
        t = _uniq(X.sample_tree(rules, rng), [0]); out = X.render(t, spec); leaves = np.array([x - 1000 for x in out if x >= 1000]); n = len(leaves)
        pos = np.empty(n, int); pos[leaves] = np.arange(n)
        disp.append(np.abs(pos - np.arange(n)).mean() / n)
        tau.append(((pos[:, None] > pos[None, :]) & (np.arange(n)[:, None] < np.arange(n)[None, :])).sum() / (n * (n - 1) / 2))
        ins.append((len(out) - n) / n)
    tm = spec.get("term_map"); tokchg = float(np.mean([tm[k] != k for k in tm])) if tm else 0.0
    return dict(displacement=float(np.mean(disp)), kendall_tau=float(np.mean(tau)), inserted=float(np.mean(ins)), token_change=tokchg)

BASE = {g: string_baselines(g) for g in X.G_LIST}

T = {}
for p in glob.glob("exp8/translator/*_d32_s0.json"):
    r = json.load(open(p)); T[r["g"]] = r
idn = T["identity"]["auc"]
gs = [g for g in X.G_LIST if g in T]
rk = lambda v: np.argsort(np.argsort(np.asarray(v, float))).astype(float)
sp = lambda x, y: float(np.corrcoef(rk(x), rk(y))[0, 1])
a = [T[g]["auc"] - idn for g in gs]
print(f"n={len(gs)} renderings with a translator result: {gs}")
for k in ("kendall_tau", "displacement", "inserted", "token_change"):
    print(f"  Spearman(a(g), {k:13s}) = {sp(a, [BASE[g][k] for g in gs]):+.2f}")

n=10 renderings with a translator result: ['relabel_new', 'relabel_perm', 'marker', 'swap5', 'rev6', 'rev5', 'rev4', 'rev45', 'rev_all', 'perm_sym']
  Spearman(a(g), kendall_tau  ) = +0.77
  Spearman(a(g), displacement ) = +0.77
  Spearman(a(g), inserted     ) = +0.36
  Spearman(a(g), token_change ) = -0.16


In [3]:
rows = sorted(zip(gs, a, [BASE[g]['kendall_tau'] for g in gs], [T[g]['heldout_acc'] for g in gs]), key=lambda r: r[1])
ra, rt = rk([r[1] for r in rows]), rk([r[2] for r in rows])
print(f"{'g':13s} {'a':>8s} {'tau':>6s} {'rank_a':>7s} {'rank_tau':>9s} {'acc':>6s}")
for (g, aa, tt, acc), x, y in zip(rows, ra, rt): print(f"{g:13s} {aa:8.4f} {tt:6.3f} {x:7.0f} {y:9.0f} {acc:6.3f}")

g                    a    tau  rank_a  rank_tau    acc
relabel_perm    0.0033  0.000       0         0  1.000
marker          0.0041  0.000       1         1  1.000
relabel_new     0.0079  0.000       2         2  1.000
rev6            0.0085  0.007       3         3  0.999
rev_all         0.0237  1.000       4         9  1.000
swap5           0.0396  0.009       5         4  0.991
rev5            0.0438  0.018       6         5  0.994
rev45           0.1125  0.057       7         7  0.983
rev4            0.1387  0.039       8         6  0.980
perm_sym        0.2685  0.595       9         8  0.940
